Copyright 2026, [AGH University of Krakow](https://www.agh.edu.pl/en) , [Institute of Telecommunications](https://tele.agh.edu.pl/)
Author: **Jaroslaw Bulat** kwant@agh.edu.pl, [LinkedIn](https://www.linkedin.com/in/jaros%C5%82aw-bu%C5%82at-30a9b191/), [WWW](https://home.agh.edu.pl/kwant/)

# **Yin Yang - Small Autoencoder**

Train a compact CNN autoencoder on the Yin Yang dataset.

## Import Libraries

In [ ]:
# fetch repository when running on Colab
!git clone https://github.com/kwanty/YinYang.git YinYang-repo
%cd YinYang-repo

import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from yinyang import generate_dataset

## Load Dataset

In [ ]:
# Option 1: Generate a fresh dataset and save it
x_train, y_train = generate_dataset(10000)
# np.savez_compressed('../data/yinyang_10k.npz', x_train=x_train, y_train=y_train)

# Option 2: Load existing dataset from file
# data = np.load('../data/yinyang_10k.npz')
# x_train = data['x_train']
# y_train = data['y_train']

print(f'Dataset shape: {x_train.shape}')
print(f'Labels shape: {y_train.shape}')

## Build CNN autoencoder for 2D latent space

Convlutional architecter is more suitable for image processing. Let's use a stack of convolutional and pooling layers.

In [ ]:
# Define image dimensions parametrically
img_size = (28, 28)
latent_dim = 2  # Small latent space

# Input layer
input_img = keras.layers.Input(shape=img_size)

# Encoder: flatten and compress to small latent space
encoder = keras.Sequential([
    keras.layers.Reshape((*img_size, 1)), # 1-channel image (grayscale)
    keras.layers.Conv2D(16, (3, 3), activation='relu', padding='same'),
    keras.layers.MaxPooling2D(2), # 14x14x16
    keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    keras.layers.MaxPooling2D(2), # 7x7x32
    keras.layers.Flatten(),
    keras.layers.Dense(4, activation='relu'),           # at least one non-linear layer
    keras.layers.Dense(latent_dim, activation='linear') # latent space (linear!)
])

# Decoder: expand from latent space and reshape
decoder = keras.Sequential([
    keras.layers.Dense(7*7*32, activation='relu'),
    keras.layers.Reshape((7, 7, 32)),
    keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    keras.layers.UpSampling2D(2),  # 14x14
    keras.layers.Conv2D(16, (3, 3), activation='relu', padding='same'),
    keras.layers.UpSampling2D(2), # 28x28
    # Output is pixel brightness; 'sigmoid' typically performs better than 'linear' here
    keras.layers.Conv2D(1, (3, 3), activation='sigmoid', padding='same'),
    keras.layers.Reshape(img_size)
])

autoencoder = decoder(encoder(input_img))

model = keras.Model(input_img, autoencoder)
model.compile(optimizer='adam', loss='mse', metrics=['MAE'])

print(model.summary(expand_nested=True))

### Train

Train the model - nothing extraordinary here.

In [ ]:
history = model.fit(
    x_train, x_train,
    epochs=50,
    batch_size=8,
    validation_split=0.2,
    verbose=1
)

### Evaluate and Visualize

Notice, MAE < 0.004 is error less then smallest difference in image brightness (assuming 256 brightness level).

In [ ]:
# Plot training history
plt.figure(figsize=(10, 4))
plt.plot(history.history['MAE'], label='Training loss')
plt.plot(history.history['val_MAE'], label='Validation loss')
plt.xlabel('Epoch')
plt.ylabel('Valid MAE')
plt.ylim(0, 0.05)
plt.legend()
plt.grid()
plt.show()

### Reconstruction Examples

In [ ]:
# Get reconstructions
reconstructed = model.predict(x_train[:10])

# Visualize original vs reconstructed
fig, axes = plt.subplots(2, 10, figsize=(15, 3))

for i in range(10):
    axes[0, i].imshow(x_train[i], cmap='gray')
    axes[0, i].set_title('Original')
    axes[0, i].axis('off')

    axes[1, i].imshow(reconstructed[i], cmap='gray')
    axes[1, i].set_title('Reconstructed')
    axes[1, i].axis('off')

plt.suptitle('Original vs Reconstructed (Small AE)', fontsize=12)
plt.tight_layout()
plt.show()

### Latent Space Analysis

Explore the 2-dimensional latent representation. It is 2D, thus we can plot it on a screen :-).

### Summary and Discussion

**Redundancy and Representation**

The input data consists of images with a size of **28x28** (784 pixels), yet each was generated from only **one control value** (a parameter ranging from 0...1) that represents the transformation state of the Yin Yang symbol.

This means the image representation is extremely redundant - all information contained within the 784 pixels essentially goes down to a single dimension. The autoencoder learns to compress this redundancy:
* **The Encoder** performs the transformation 28x28 → latent_dim.
* **The Decoder** reconstructs this relationship in reverse: latent_dim → 28x28.

**Spontaneous Regularization of the Latent Space**


Even though we defined the latent space as **2D** and did not impose any additional constraints on it (unlike a VAE), we observe a fascinating phenomenon in the visualization:

1. **A continuous line, not a cloud:** The points do not fill the 2D surface but instead arrange themselves into a continuous, winding line. This confirms that the model discovered the actual structure of the data - the fact that they depend on only one parameter.
2. **No intersections:** The curve in the latent space never intersects itself. This comes from the nature of the problem - if the line were to intersect, one point (small region) in the latent space would have to correspond to two different images, which would make correct reconstruction impossible for the decoder (it would create ambiguity).
3. **Arbitrariness of shape:** The shape of this curve (whether it is a spiral, an arc, or a loop) varies with each training run. This happens because there are no restrictions forcing a specific layout (e.g., linearity) - only maintaining topological continuity matters.
4. **Curvature:** The representation is a curve rather than a straight line because the convolutional layers and non-linear activation functions map the image space in a non-linear way, optimizing the MSE minimization process.

In [ ]:
# Create an encoder-only model to extract latent vectors
# Defining the autoencoder in two parts makes it easy to split into halves
encoder_model = keras.Model(input_img, encoder(input_img))

# Encode the dataset
latent_vectors = encoder_model.predict(x_train)

# Visualize the latent space with colors based on ground truth labels
plt.figure(figsize=(10, 8))
scatter = plt.scatter(latent_vectors[:, 0], latent_vectors[:, 1], c=y_train, cmap='viridis', alpha=0.5, s=2)
plt.colorbar(scatter, label='Class / Label')
plt.title('Latent Space Visualization (2D)')
plt.xlabel('Latent Dimension 1')
plt.ylabel('Latent Dimension 2')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

print(f'Latent vectors shape: {latent_vectors.shape}')

### Latent Space Manifold
Visualize how the decoder maps different regions of the 2D latent space back to image space.

In [ ]:
n = 16  # figure with 15x15 digits
img_size_val = 28
figure = np.zeros((img_size_val * n, img_size_val * n))

# Define the range of the latent space to explore
# Based on the previous scatter plot, we adjust these limits
x_min, x_max = latent_vectors[:, 0].min(), latent_vectors[:, 0].max()
y_min, y_max = latent_vectors[:, 1].min(), latent_vectors[:, 1].max()

grid_x = np.linspace(x_min, x_max, n)
grid_y = np.linspace(y_min, y_max, n)[::-1]

for i, yi in enumerate(grid_y):
    for j, xi in enumerate(grid_x):
        z_sample = np.array([[xi, yi]])
        # Use an decoder-only model to extract image from latent space
        # Defining the autoencoder in two parts makes it easy to split into halves
        x_decoded = decoder.predict(z_sample, verbose=0)
        digit = x_decoded[0].reshape(img_size_val, img_size_val)
        figure[i * img_size_val: (i + 1) * img_size_val, j * img_size_val: (j + 1) * img_size_val] = digit

plt.figure(figsize=(12, 12))
start_range = img_size_val // 2
end_range = n * img_size_val + start_range
pixel_range = np.arange(start_range, end_range, img_size_val)
sample_range_x = np.round(grid_x, 1)
sample_range_y = np.round(grid_y, 1)

plt.xticks(pixel_range, sample_range_x)
plt.yticks(pixel_range, sample_range_y)
plt.xlabel("Latent Dimension 1")
plt.ylabel("Latent Dimension 2")
plt.imshow(figure, cmap='gray')
plt.title("Decoded Manifold from Latent Space")
plt.show()

## Build CNN autoencoder for 1D latent space

Since the problem is 1D, lets build and test autoencoder with `latent_dim = 1`. The same data, problem, architecture, the only difference is in a size of the latent space.

In [ ]:
# Define image dimensions parametrically
img_size = (28, 28)
latent_dim = 1  # Small latent space

# Input layer
input_img = keras.layers.Input(shape=img_size)

# Encoder: flatten and compress to small latent space
encoder = keras.Sequential([
    keras.layers.Reshape((*img_size, 1)), # 1-channel image (grayscale)
    keras.layers.Conv2D(16, (3, 3), activation='relu', padding='same'),
    keras.layers.MaxPooling2D(2), # 14x14x16
    keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    keras.layers.MaxPooling2D(2), # 7x7x32
    keras.layers.Flatten(),
    keras.layers.Dense(4, activation='relu'),           # at least one non-linear layer
    keras.layers.Dense(latent_dim, activation='linear') # latent space (linear!)
])

# Decoder: expand from latent space and reshape
decoder = keras.Sequential([
    keras.layers.Dense(7*7*32, activation='relu'),
    keras.layers.Reshape((7, 7, 32)),
    keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    keras.layers.UpSampling2D(2),  # 14x14
    keras.layers.Conv2D(16, (3, 3), activation='relu', padding='same'),
    keras.layers.UpSampling2D(2), # 28x28
    # Output is pixel brightness; 'sigmoid' typically performs better than 'linear' here
    keras.layers.Conv2D(1, (3, 3), activation='sigmoid', padding='same'),
    keras.layers.Reshape(img_size)
])

autoencoder = decoder(encoder(input_img))

model = keras.Model(input_img, autoencoder)
model.compile(optimizer='adam', loss='mse', metrics=['MAE'])

print(model.summary(expand_nested=True))

### Train

Train the model - nothing extraordinary here.

In [ ]:
history = model.fit(
    x_train, x_train,
    epochs=50,
    batch_size=8,
    validation_split=0.2,
    verbose=1
)

### Evaluate and Visualize

Notice, MAE < 0.004 is error less then smallest difference in image brightness (assuming 256 brightness level).

In [ ]:
# Plot training history
plt.figure(figsize=(10, 4))
plt.plot(history.history['MAE'], label='Training loss')
plt.plot(history.history['val_MAE'], label='Validation loss')
plt.xlabel('Epoch')
plt.ylabel('Valid MAE')
plt.ylim(0, 0.05)
plt.legend()
plt.grid()
plt.show()

### Reconstruction Examples

In [ ]:
# Get reconstructions
reconstructed = model.predict(x_train[:10])

# Visualize original vs reconstructed
fig, axes = plt.subplots(2, 10, figsize=(15, 3))

for i in range(10):
    axes[0, i].imshow(x_train[i], cmap='gray')
    axes[0, i].set_title('Original')
    axes[0, i].axis('off')

    axes[1, i].imshow(reconstructed[i], cmap='gray')
    axes[1, i].set_title('Reconstructed')
    axes[1, i].axis('off')

plt.suptitle('Original vs Reconstructed (Small AE)', fontsize=12)
plt.tight_layout()
plt.show()